# CSE570 – Machine Learning with Python
## Unit II: Exploratory Data Analysis and Feature Engineering

This notebook follows the Unit-II lecture-note sequence:

**Load Dataset → Understand Data → Missing Values → Duplicates → Visualization → Correlation → Outliers → Feature Selection → Feature Transformation → Class Imbalance → Preprocessing Pipeline → Safe Model Training**

### Main learning goals
- Perform Exploratory Data Analysis (EDA)
- Inspect missing and duplicate data
- Visualize numerical and categorical variables
- Analyze correlations
- Detect outliers using IQR
- Apply filter, wrapper, and embedded feature-selection methods
- Transform numerical and categorical variables
- Handle class imbalance
- Build preprocessing pipelines
- Avoid data leakage

> **Dataset expected:** `CSE570_Unit2_Customer_Churn_Dataset.csv`  
> Keep the CSV file in the same folder as this notebook before running it.

## 1. Import Required Libraries

We first import the libraries used throughout the notebook.

Some libraries are imported again later near the concept where they are used so that students can associate each library with its purpose.

In [ ]:
# =========================
# BASIC LIBRARY IMPORTS
# =========================

# pandas is used for reading, cleaning, and manipulating tabular data.
import pandas as pd

# NumPy provides mathematical and numerical operations.
import numpy as np

# matplotlib is used for plotting graphs.
import matplotlib.pyplot as plt

# Display options make DataFrame output easier to inspect.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")

## 2. Load the Customer Churn Dataset

The lecture notes use the file:

`CSE570_Unit2_Customer_Churn_Dataset.csv`

The following cell checks whether the file exists before attempting to load it.

In [ ]:
from pathlib import Path

# Name of the dataset used in the Unit-II lecture notes.
DATA_FILE = Path("CSE570_Unit2_Customer_Churn_Dataset.csv")

# Check whether the file is available in the notebook folder.
if DATA_FILE.exists():
    # Read the CSV file and store it in a pandas DataFrame named df.
    df = pd.read_csv(DATA_FILE)

    print("Dataset loaded successfully.")
    print("Shape:", df.shape)
else:
    # Stop with a clear message instead of producing a confusing FileNotFoundError later.
    raise FileNotFoundError(
        f"{DATA_FILE} was not found. "
        "Please place the CSV file in the same folder as this notebook."
    )

## 3. Exploratory Data Analysis (EDA)

EDA helps us understand the dataset before applying machine-learning algorithms.

We inspect:
- first few records,
- number of rows and columns,
- column names and data types,
- missing values,
- descriptive statistics.

In [ ]:
# Display the first 5 rows of the dataset.
print("FIRST FIVE ROWS")
display(df.head())

# Display the dimensions of the dataset.
# shape returns: (number_of_rows, number_of_columns)
print("\nDATASET SHAPE")
print(df.shape)

# Display column names.
print("\nCOLUMN NAMES")
print(df.columns.tolist())

# info() shows:
# - column names,
# - non-null counts,
# - data types,
# - approximate memory usage.
print("\nDATASET INFORMATION")
df.info()

# describe() produces summary statistics for numerical columns.
print("\nNUMERICAL SUMMARY")
display(df.describe())

## 4. Missing Values

A missing value means that information is unavailable for a particular observation.

Typical treatments:
- numerical feature → mean or median,
- categorical feature → mode,
- very high missingness → consider removing the feature,
- meaningful missingness → consider a separate indicator/category.

In [ ]:
# Count missing values in every column.
missing_count = df.isnull().sum()

# Calculate missing-value percentages.
missing_percentage = df.isnull().mean() * 100

# Combine both into one table.
missing_summary = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percentage": missing_percentage
})

# Show only columns that contain at least one missing value.
missing_summary = missing_summary[missing_summary["Missing_Count"] > 0]

print("MISSING-VALUE SUMMARY")
display(missing_summary.sort_values("Missing_Percentage", ascending=False))

In [ ]:
# Example from the lecture notes:
# Replace missing Age values with the median Age.
#
# Median is preferred in this example because it is less affected
# by extreme values/outliers than the mean.

if "Age" in df.columns:
    print("Missing Age values before treatment:", df["Age"].isnull().sum())

    # Compute the median only from the available Age values.
    age_median = df["Age"].median()

    # Fill missing Age values with the median.
    df["Age"] = df["Age"].fillna(age_median)

    print("Median Age used:", age_median)
    print("Missing Age values after treatment:", df["Age"].isnull().sum())
else:
    print("Column 'Age' is not present, so this example is skipped.")

## 5. Duplicate Records

Duplicate records may result from repeated data entry or dataset merging.

**Important:** duplicates should not be removed blindly. Two identical transaction rows, for example, may represent two valid transactions.

In [ ]:
# Count exact duplicate rows.
duplicate_count = df.duplicated().sum()

print("Number of exact duplicate rows:", duplicate_count)

# Display duplicate rows if any exist.
if duplicate_count > 0:
    display(df[df.duplicated(keep=False)].head(20))

In [ ]:
# Remove exact duplicate rows.
#
# We use copy() so that the cleaned DataFrame is clearly separated
# from any previous view/reference.

rows_before = len(df)

df = df.drop_duplicates().copy()

rows_after = len(df)

print("Rows before duplicate removal:", rows_before)
print("Rows after duplicate removal :", rows_after)
print("Rows removed                :", rows_before - rows_after)

# 6. Data Visualization

Choose the plot according to the **type of variable** and the **question being asked**.

| Question | Recommended Plot |
|---|---|
| Distribution of one numerical feature | Histogram |
| Possible outliers | Boxplot |
| Frequency of categories | Bar plot |
| Relationship between two numerical features | Scatter plot |
| Numerical feature across categories | Boxplot |
| Category vs target | Grouped bar |
| Many numerical relationships | Correlation heatmap |
| Target balance | Count/bar plot |

## 6.1 Histogram — Numerical Distribution

In [ ]:
# Draw a histogram for Age.

if "Age" in df.columns:
    plt.figure(figsize=(8, 4))

    # bins=20 divides the full Age range into 20 intervals.
    df["Age"].dropna().hist(bins=20)

    plt.xlabel("Age")
    plt.ylabel("Frequency")
    plt.title("Distribution of Customer Age")
    plt.tight_layout()
    plt.show()
else:
    print("Column 'Age' is not present, so the histogram is skipped.")

### Compare several numerical features

In [ ]:
# Numerical columns used in the lecture notes.
requested_hist_features = [
    "Age",
    "TenureMonths",
    "MonthlyCharges",
    "InternetUsageGB",
    "AnnualIncome"
]

# Keep only features that actually exist in the dataset.
hist_features = [col for col in requested_hist_features if col in df.columns]

if hist_features:
    # pandas automatically creates one histogram per selected feature.
    df[hist_features].hist(figsize=(12, 8), bins=20)

    # Adjust spacing so axes and titles do not overlap.
    plt.tight_layout()
    plt.show()
else:
    print("None of the requested numerical features are available.")

## 6.2 Boxplot — Distribution and Potential Outliers

In [ ]:
if "MonthlyCharges" in df.columns:
    plt.figure(figsize=(8, 3))

    # dropna() ensures that only valid numerical observations are plotted.
    # vert=False creates a horizontal boxplot.
    plt.boxplot(
        df["MonthlyCharges"].dropna(),
        vert=False
    )

    plt.xlabel("Monthly Charges")
    plt.title("Boxplot of Monthly Charges")
    plt.tight_layout()
    plt.show()
else:
    print("Column 'MonthlyCharges' is not present.")

## 6.3 Bar Plot — Categorical Variable

In [ ]:
if "ContractType" in df.columns:
    # value_counts() counts how many customers belong to each category.
    df["ContractType"].value_counts().plot(kind="bar", figsize=(8, 4))

    plt.xlabel("Contract Type")
    plt.ylabel("Number of Customers")
    plt.title("Customers by Contract Type")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Column 'ContractType' is not present.")

## 6.4 Target-Class Visualization

Before classification, inspect whether the target classes are balanced.

In [ ]:
if "Churn" in df.columns:
    # sort_index() keeps labels such as 0 and 1 in logical order.
    df["Churn"].value_counts().sort_index().plot(kind="bar", figsize=(6, 4))

    plt.xlabel("Churn")
    plt.ylabel("Number of Customers")
    plt.title("Churn Class Distribution")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    print("Class counts:")
    display(df["Churn"].value_counts().sort_index())

    print("Class proportions:")
    display(df["Churn"].value_counts(normalize=True).sort_index())
else:
    print("Target column 'Churn' is not present.")

## 6.5 Grouped Bar Plot — Categorical Feature vs Target

In [ ]:
if {"ContractType", "Churn"}.issubset(df.columns):
    # Build a frequency table between ContractType and Churn.
    contract_churn_counts = pd.crosstab(
        df["ContractType"],
        df["Churn"]
    )

    print("COUNT TABLE")
    display(contract_churn_counts)

    # Plot counts as grouped bars.
    contract_churn_counts.plot(kind="bar", figsize=(9, 5))
    plt.ylabel("Number of Customers")
    plt.title("Contract Type vs Churn")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Required columns 'ContractType' and/or 'Churn' are missing.")

In [ ]:
if {"ContractType", "Churn"}.issubset(df.columns):
    # normalize="index" converts each contract-type row into proportions.
    contract_churn_percentage = (
        pd.crosstab(
            df["ContractType"],
            df["Churn"],
            normalize="index"
        ) * 100
    )

    print("PERCENTAGE TABLE")
    display(contract_churn_percentage.round(2))

## 6.6 Scatter Plot — Relationship Between Numerical Features

In [ ]:
if {"TenureMonths", "TotalCharges"}.issubset(df.columns):
    plt.figure(figsize=(7, 5))

    plt.scatter(
        df["TenureMonths"],
        df["TotalCharges"],
        alpha=0.5
    )

    plt.xlabel("Tenure Months")
    plt.ylabel("Total Charges")
    plt.title("Tenure vs Total Charges")
    plt.tight_layout()
    plt.show()
else:
    print("Required columns 'TenureMonths' and/or 'TotalCharges' are missing.")

### Scatter Plot with Target Classes

In [ ]:
if {"TenureMonths", "TotalCharges", "Churn"}.issubset(df.columns):
    plt.figure(figsize=(8, 5))

    # Plot each target class separately on the same graph.
    for churn_class in sorted(df["Churn"].dropna().unique()):
        subset = df[df["Churn"] == churn_class]

        plt.scatter(
            subset["TenureMonths"],
            subset["TotalCharges"],
            alpha=0.5,
            label=f"Churn = {churn_class}"
        )

    plt.xlabel("Tenure Months")
    plt.ylabel("Total Charges")
    plt.title("Tenure vs Total Charges by Churn")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Required columns are missing.")

## 6.7 Numerical Feature vs Categorical Group

Compare `MonthlyCharges` between retained and churned customers.

In [ ]:
if {"MonthlyCharges", "Churn"}.issubset(df.columns):
    # Select MonthlyCharges for customers who did not churn.
    group0 = df[df["Churn"] == 0]["MonthlyCharges"].dropna()

    # Select MonthlyCharges for customers who churned.
    group1 = df[df["Churn"] == 1]["MonthlyCharges"].dropna()

    plt.figure(figsize=(7, 5))

    plt.boxplot(
        [group0, group1],
        tick_labels=["Retained", "Churned"]
    )

    plt.ylabel("Monthly Charges")
    plt.title("Monthly Charges by Churn Status")
    plt.tight_layout()
    plt.show()
else:
    print("Required columns 'MonthlyCharges' and/or 'Churn' are missing.")

## 6.8 Correlation Heatmap

In [ ]:
# Select only numerical columns.
numeric_df_for_corr = df.select_dtypes(include="number")

if numeric_df_for_corr.shape[1] >= 2:
    # Pearson correlation is the default used by DataFrame.corr().
    corr = numeric_df_for_corr.corr()

    plt.figure(figsize=(10, 7))

    # imshow() displays the correlation matrix as an image.
    plt.imshow(corr, aspect="auto")

    # Add a scale to interpret correlation strength.
    plt.colorbar(label="Correlation")

    # Add feature names to both axes.
    plt.xticks(
        range(len(corr.columns)),
        corr.columns,
        rotation=45,
        ha="right"
    )

    plt.yticks(
        range(len(corr.index)),
        corr.index
    )

    plt.title("Correlation Matrix")
    plt.tight_layout()
    plt.show()
else:
    print("At least two numerical columns are required.")

## 6.9 Visualizing Missing Values

In [ ]:
# Recalculate percentages because some earlier cells may already have
# filled the missing Age values.
missing_percentage = (df.isnull().mean() * 100).sort_values(ascending=False)

# Keep only columns containing missing values.
missing_to_plot = missing_percentage[missing_percentage > 0]

if len(missing_to_plot) > 0:
    missing_to_plot.plot(kind="bar", figsize=(9, 4))

    plt.ylabel("Missing Values (%)")
    plt.title("Missing Values by Feature")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values are currently present in the dataset.")

# 7. Correlation Analysis

Correlation measures the strength and direction of association between variables.

- **Pearson**: mainly linear relationships between numerical variables.
- **Spearman**: useful for monotonic relationships, ordinal data, non-normal data, or data affected by outliers.

> Correlation does **not** prove causation.

In [ ]:
numeric_df = df.select_dtypes(include="number")

# Pearson correlation matrix.
pearson_corr = numeric_df.corr(method="pearson")

print("PEARSON CORRELATION")
display(pearson_corr)

In [ ]:
# Spearman correlation matrix.
spearman_corr = numeric_df.corr(method="spearman")

print("SPEARMAN CORRELATION")
display(spearman_corr)

# 8. Outlier Detection using the IQR Method

The Interquartile Range is:

**IQR = Q3 − Q1**

Potential outlier limits:

- Lower = Q1 − 1.5 × IQR
- Upper = Q3 + 1.5 × IQR

In [ ]:
if "MonthlyCharges" in df.columns:
    # Calculate Q1: 25th percentile.
    q1 = df["MonthlyCharges"].quantile(0.25)

    # Calculate Q3: 75th percentile.
    q3 = df["MonthlyCharges"].quantile(0.75)

    # Calculate the Interquartile Range.
    iqr = q3 - q1

    # Calculate the lower and upper boundaries.
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    # Select observations lying outside the boundaries.
    outliers = df[
        (df["MonthlyCharges"] < lower) |
        (df["MonthlyCharges"] > upper)
    ]

    print("Q1:", q1)
    print("Q3:", q3)
    print("IQR:", iqr)
    print("Lower boundary:", lower)
    print("Upper boundary:", upper)
    print("Number of potential outliers:", len(outliers))

    display(outliers.head(20))
else:
    print("Column 'MonthlyCharges' is not present.")

### Optional: Capping Extreme Values

The lecture notes list capping as one possible treatment.

This cell does **not** overwrite the original feature. Instead, it creates a new demonstration column so that the original data remains available for comparison.

In [ ]:
if "MonthlyCharges" in df.columns:
    # Create a copy of the original feature.
    df["MonthlyCharges_Capped"] = df["MonthlyCharges"].copy()

    # Values below lower are replaced by lower.
    # Values above upper are replaced by upper.
    df["MonthlyCharges_Capped"] = df["MonthlyCharges_Capped"].clip(
        lower=lower,
        upper=upper
    )

    print("Original vs capped values:")
    display(df[["MonthlyCharges", "MonthlyCharges_Capped"]].head(10))

# 9. Feature Selection

Feature selection keeps the most useful original input variables and removes irrelevant, redundant, noisy, or unnecessarily complex features.

Three main families:
1. **Filter methods**
2. **Wrapper methods**
3. **Embedded methods**

## 9.1 Remove Identifier Features and Separate Target

In [ ]:
# Create a feature matrix X and target vector y.
#
# CustomerID is removed if present because it is an identifier.
# Churn is removed from X because it is the target.

if "Churn" not in df.columns:
    raise KeyError("The target column 'Churn' is required for the feature-selection examples.")

columns_to_drop = ["Churn"]

if "CustomerID" in df.columns:
    columns_to_drop.append("CustomerID")

X_all = df.drop(columns=columns_to_drop)
y = df["Churn"]

print("Feature matrix shape:", X_all.shape)
print("Target shape:", y.shape)
print("Dropped columns:", columns_to_drop)

## 9.2 Constant and Near-Constant Features

In [ ]:
# nunique() counts the number of unique values in each column.
unique_counts = X_all.nunique(dropna=False).sort_values()

print("NUMBER OF UNIQUE VALUES PER FEATURE")
display(unique_counts)

# A constant feature has only one unique value.
constant_features = unique_counts[unique_counts <= 1].index.tolist()

print("Constant features:", constant_features)

## 9.3 Correlation-Based Redundancy Check

In [ ]:
# Select only numerical input features.
numeric_predictors = X_all.select_dtypes(include="number")

# Calculate the absolute correlation matrix.
# abs() makes both strong positive and strong negative correlations large.
abs_corr = numeric_predictors.corr().abs()

display(abs_corr)

In [ ]:
# Example: list strongly correlated predictor pairs.
#
# Threshold 0.80 is used here only as an inspection threshold.
# The lecture notes emphasize that correlated features should not be
# removed automatically without considering domain meaning and model type.

threshold = 0.80

strong_pairs = []

for i in range(len(abs_corr.columns)):
    for j in range(i + 1, len(abs_corr.columns)):
        value = abs_corr.iloc[i, j]

        if pd.notna(value) and value >= threshold:
            strong_pairs.append({
                "Feature_1": abs_corr.columns[i],
                "Feature_2": abs_corr.columns[j],
                "Absolute_Correlation": value
            })

strong_pairs_df = pd.DataFrame(strong_pairs)

if len(strong_pairs_df) > 0:
    display(strong_pairs_df.sort_values("Absolute_Correlation", ascending=False))
else:
    print(f"No numerical feature pairs have absolute correlation >= {threshold}.")

## 9.4 Prepare Numerical Features for Feature-Selection Demonstrations

The lecture notes demonstrate selection using these numerical features:

- Age
- TenureMonths
- MonthlyCharges
- SupportCalls
- SatisfactionScore
- InternetUsageGB
- AnnualIncome

In [ ]:
# Feature list from the lecture notes.
requested_features = [
    "Age",
    "TenureMonths",
    "MonthlyCharges",
    "SupportCalls",
    "SatisfactionScore",
    "InternetUsageGB",
    "AnnualIncome"
]

# Keep only columns that are actually available.
features = [col for col in requested_features if col in df.columns]

if len(features) < 2:
    raise ValueError(
        "At least two of the numerical lecture-note features are needed "
        "for the following feature-selection demonstrations."
    )

# Create the numerical feature matrix.
X = df[features].copy()

# Fill missing numerical values with each column's median.
# Feature-selection algorithms generally cannot accept NaN directly.
X = X.fillna(X.median(numeric_only=True))

# Keep target separately.
y = df["Churn"]

print("Features used:")
print(features)

display(X.head())

## 9.5 Filter Method — SelectKBest with ANOVA F-test

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# Keep at most 4 features.
# If fewer than 4 are available, keep all available features.
k = min(4, X.shape[1])

selector = SelectKBest(
    score_func=f_classif,
    k=k
)

# Learn an ANOVA F-score for every feature.
selector.fit(X, y)

# Extract selected feature names.
selected_features_anova = X.columns[selector.get_support()]

print("Selected features using ANOVA F-test:")
print(selected_features_anova.tolist())

In [ ]:
# Build a result table for easier interpretation.
anova_scores = pd.DataFrame({
    "Feature": X.columns,
    "F_Score": selector.scores_,
    "Selected": selector.get_support()
})

anova_scores = anova_scores.sort_values(
    "F_Score",
    ascending=False
)

display(anova_scores)

## 9.6 Filter Method — Mutual Information

In [ ]:
from sklearn.feature_selection import mutual_info_classif

# Mutual Information can capture broader dependency than simple
# linear correlation.
mi_scores = mutual_info_classif(
    X,
    y,
    random_state=42
)

mi = pd.Series(
    mi_scores,
    index=X.columns,
    name="Mutual_Information"
).sort_values(ascending=False)

display(mi.to_frame())

## 9.7 Wrapper Method — Recursive Feature Elimination (RFE)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

# Logistic Regression is used as the estimator that judges
# which features should be retained.
rfe_model = LogisticRegression(
    max_iter=1000
)

# Keep at most 4 features.
rfe_k = min(4, X.shape[1])

rfe = RFE(
    estimator=rfe_model,
    n_features_to_select=rfe_k
)

# Run recursive feature elimination.
rfe.fit(X, y)

rfe_result = pd.DataFrame({
    "Feature": X.columns,
    "Selected": rfe.support_,
    "Ranking": rfe.ranking_
})

display(rfe_result.sort_values(["Selected", "Ranking"], ascending=[False, True]))

## 9.8 Embedded Method — Random Forest Feature Importance

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create a Random Forest with 200 trees.
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

# Train the forest.
rf.fit(X, y)

# Extract model-based feature importance.
importance = pd.Series(
    rf.feature_importances_,
    index=X.columns,
    name="Importance"
).sort_values(ascending=False)

display(importance.to_frame())

In [ ]:
# Visualize Random Forest feature importance.
importance.sort_values().plot(
    kind="barh",
    figsize=(8, 5)
)

plt.xlabel("Importance")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

## 9.9 Embedded Method — L1 Regularization

In [ ]:
from sklearn.linear_model import LogisticRegression

# L1 regularization can shrink some coefficients to exactly zero.
l1_model = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    max_iter=1000
)

l1_model.fit(X, y)

# Extract learned coefficients.
coefficients = pd.Series(
    l1_model.coef_[0],
    index=X.columns,
    name="L1_Coefficient"
)

# Add absolute magnitude to make comparison easier.
l1_table = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": coefficients.values,
    "Absolute_Coefficient": np.abs(coefficients.values)
}).sort_values("Absolute_Coefficient", ascending=False)

display(l1_table)

## 9.10 Compare Feature-Selection Results

In [ ]:
# Create one comparison table from the different methods.

comparison = pd.DataFrame(index=X.columns)

# ANOVA F-score
comparison["ANOVA_F_Score"] = pd.Series(
    selector.scores_,
    index=X.columns
)

# Mutual Information
comparison["Mutual_Information"] = pd.Series(
    mi_scores,
    index=X.columns
)

# RFE ranking
comparison["RFE_Rank"] = pd.Series(
    rfe.ranking_,
    index=X.columns
)

# Random Forest importance
comparison["RF_Importance"] = pd.Series(
    rf.feature_importances_,
    index=X.columns
)

# L1 coefficient
comparison["L1_Coefficient"] = pd.Series(
    l1_model.coef_[0],
    index=X.columns
)

display(comparison)

# 10. Feature Transformation

Feature transformation changes the representation or scale of a feature.

The lecture notes cover:
- Standardization
- Min-Max scaling
- Log transformation
- One-Hot Encoding

## 10.1 Standardization

In [ ]:
from sklearn.preprocessing import StandardScaler

scale_columns = [
    col for col in ["Age", "MonthlyCharges"]
    if col in df.columns
]

if scale_columns:
    scaler = StandardScaler()

    # Median imputation is used before scaling to avoid NaN errors.
    data_for_scaling = df[scale_columns].copy()
    data_for_scaling = data_for_scaling.fillna(
        data_for_scaling.median(numeric_only=True)
    )

    standardized = scaler.fit_transform(data_for_scaling)

    standardized_df = pd.DataFrame(
        standardized,
        columns=[f"{col}_Standardized" for col in scale_columns],
        index=df.index
    )

    display(standardized_df.head())
else:
    print("Required scaling columns are not available.")

## 10.2 Min-Max Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler

if scale_columns:
    minmax_scaler = MinMaxScaler()

    data_for_minmax = df[scale_columns].copy()
    data_for_minmax = data_for_minmax.fillna(
        data_for_minmax.median(numeric_only=True)
    )

    minmax_values = minmax_scaler.fit_transform(data_for_minmax)

    minmax_df = pd.DataFrame(
        minmax_values,
        columns=[f"{col}_MinMax" for col in scale_columns],
        index=df.index
    )

    display(minmax_df.head())

## 10.3 Log Transformation

In [ ]:
if "AnnualIncome" in df.columns:
    # log1p(x) calculates log(1 + x).
    # It is safer than log(x) when x may contain zero.
    df["LogIncome"] = np.log1p(df["AnnualIncome"])

    display(df[["AnnualIncome", "LogIncome"]].head(10))
else:
    print("Column 'AnnualIncome' is not present.")

## 10.4 One-Hot Encoding

In [ ]:
if "ContractType" in df.columns:
    # One-hot encoding creates a separate 0/1 column for every category.
    contract_encoded = pd.get_dummies(
        df["ContractType"],
        prefix="ContractType"
    )

    display(contract_encoded.head())
else:
    print("Column 'ContractType' is not present.")

# 11. Data Imbalance Handling

A dataset is imbalanced when one target class has many more samples than another.

For imbalanced classification, accuracy alone may be misleading.

Other useful metrics include:
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion Matrix
- Balanced Accuracy

In [ ]:
# Show class counts and proportions.

print("CLASS COUNTS")
display(y.value_counts().sort_index())

print("CLASS PROPORTIONS")
display(y.value_counts(normalize=True).sort_index())

## 11.1 Split Before Applying SMOTE

**Important:** SMOTE should be applied only to the training data.

Applying it before train-test splitting can cause data leakage.

In [ ]:
from sklearn.model_selection import train_test_split

# Split the numerical feature-selection dataset for a simple SMOTE demonstration.
X_train_fs, X_test_fs, y_train_fs, y_test_fs = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train_fs.shape)
print("Testing shape :", X_test_fs.shape)

print("\nTraining class counts BEFORE SMOTE:")
display(y_train_fs.value_counts().sort_index())

## 11.2 SMOTE

If `imbalanced-learn` is not installed, run:

`pip install imbalanced-learn`

from your environment/terminal, or uncomment the installation line below in Colab/Jupyter.

In [ ]:
# Uncomment only if imbalanced-learn is not installed:
# !pip install imbalanced-learn

try:
    from imblearn.over_sampling import SMOTE

    # Create the SMOTE object.
    smote = SMOTE(random_state=42)

    # Apply SMOTE ONLY to training data.
    X_resampled, y_resampled = smote.fit_resample(
        X_train_fs,
        y_train_fs
    )

    print("Training class counts AFTER SMOTE:")
    display(pd.Series(y_resampled).value_counts().sort_index())

except ImportError:
    print(
        "imbalanced-learn is not installed. "
        "Install it using: pip install imbalanced-learn"
    )

## 11.3 Class Weighting

In [ ]:
from sklearn.linear_model import LogisticRegression

# class_weight="balanced" gives more importance to the minority class
# according to the observed class frequencies.
balanced_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)

balanced_model.fit(X_train_fs, y_train_fs)

print("Class-weighted Logistic Regression trained successfully.")

# 12. Data Preprocessing Pipeline

A preprocessing pipeline can combine:

**Missing-value treatment → Encoding → Scaling → Model**

Advantages:
- reduces data leakage risk,
- makes preprocessing reusable,
- applies the same preprocessing to training and testing data,
- supports cleaner deployment.

## 12.1 Define Input Features and Target for the Full Pipeline

In [ ]:
# Start from the original cleaned DataFrame.

target_column = "Churn"

# Remove target from the input matrix.
# Remove CustomerID if available because it is an identifier.
drop_for_model = [target_column]

if "CustomerID" in df.columns:
    drop_for_model.append("CustomerID")

X_pipeline = df.drop(columns=drop_for_model).copy()
y_pipeline = df[target_column].copy()

# Detect numerical and categorical features automatically.
numeric_features = X_pipeline.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_pipeline.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

## 12.2 Numerical Preprocessing Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    # Step 1: fill missing numerical values using the median.
    ("imputer", SimpleImputer(strategy="median")),

    # Step 2: standardize numerical features.
    ("scaler", StandardScaler())
])

print(numeric_pipeline)

## 12.3 Categorical Preprocessing Pipeline

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline([
    # Step 1: fill missing categorical values using the most frequent category.
    ("imputer", SimpleImputer(strategy="most_frequent")),

    # Step 2: convert categories into one-hot encoded numerical columns.
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

print(categorical_pipeline)

## 12.4 Combine Numerical and Categorical Pipelines

In [ ]:
from sklearn.compose import ColumnTransformer

transformers = []

# Add numerical preprocessing only if numerical columns are available.
if numeric_features:
    transformers.append(
        ("num", numeric_pipeline, numeric_features)
    )

# Add categorical preprocessing only if categorical columns are available.
if categorical_features:
    transformers.append(
        ("cat", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers
)

print(preprocessor)

# 13. Data Leakage — Correct Train/Test Workflow

The correct order is:

1. Separate input and target.
2. Split into training and testing data.
3. Fit preprocessing **only on the training data**.
4. Transform test data using the preprocessing learned from training data.
5. Train and evaluate the model.

In [ ]:
from sklearn.model_selection import train_test_split

# Split BEFORE fitting preprocessing objects.
X_train, X_test, y_train, y_test = train_test_split(
    X_pipeline,
    y_pipeline,
    test_size=0.20,
    random_state=42,
    stratify=y_pipeline
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

## 13.1 Build Complete Preprocessing + Model Pipeline

The lecture notes end with the workflow reaching model training.  
Here we connect the already-defined preprocessing steps to Logistic Regression so the notebook can demonstrate the complete safe flow.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Create one complete machine-learning workflow.
pipeline = Pipeline([
    # Preprocessing is learned only from training data when pipeline.fit() is called.
    ("preprocessor", preprocessor),

    # Final classification model.
    ("model", LogisticRegression(max_iter=1000))
])

# Fit the entire workflow only on training data.
pipeline.fit(X_train, y_train)

print("Complete pipeline trained successfully.")

## 13.2 Make Predictions

In [ ]:
# Generate predictions for previously unseen test data.
y_pred = pipeline.predict(X_test)

print("First 20 predictions:")
print(y_pred[:20])

## 13.3 Evaluate the Model

This final evaluation cell is included to confirm that the complete preprocessing pipeline and classification workflow execute correctly.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Basic classification metrics.
accuracy = accuracy_score(y_test, y_pred)
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)

# zero_division=0 prevents warnings if a class receives no predicted samples.
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Accuracy          :", round(accuracy, 4))
print("Balanced Accuracy :", round(balanced_accuracy, 4))
print("Precision         :", round(precision, 4))
print("Recall            :", round(recall, 4))
print("F1-score          :", round(f1, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# ROC-AUC requires probability/decision scores and both target classes.
if hasattr(pipeline, "predict_proba") and len(np.unique(y_test)) == 2:
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    print("ROC-AUC           :", round(auc, 4))

# 14. Complete Unit-II Workflow Recap

The practical workflow implemented in this notebook is:

**Load Dataset**  
↓  
**Understand Data**  
↓  
**Check Missing Values**  
↓  
**Check Duplicates**  
↓  
**Visualize Data**  
↓  
**Correlation Analysis**  
↓  
**Detect Outliers**  
↓  
**Select Features**  
↓  
**Transform Features**  
↓  
**Check Class Imbalance**  
↓  
**Build Preprocessing Pipeline**  
↓  
**Train Model Safely**

### Key reminders

- Do not remove duplicates blindly.
- A boxplot identifies potential outliers; it does not prove that they are errors.
- Correlation does not imply causation.
- Do not remove correlated features automatically.
- Feature selection and feature transformation are different.
- Apply SMOTE only to training data.
- Split the data before fitting scalers, imputers, encoders, or other preprocessing steps.
- Use pipelines to reduce preprocessing inconsistency and leakage risk.

# 15. Practice Questions

1. What is EDA and why is it required?
2. Differentiate Pearson and Spearman correlation.
3. Explain the IQR method for outlier detection.
4. What is feature selection?
5. Differentiate filter, wrapper, and embedded methods.
6. Why is scaling required?
7. What is class imbalance?
8. Why can accuracy be misleading for imbalanced datasets?
9. What is SMOTE?
10. What is a preprocessing pipeline?
11. What is data leakage?
12. Why should preprocessing be fitted only on training data?